# Few-shot In-context Learning Validation Experiment

This notebook runs a few-shot in-context learning validation experiment for binary tweet classification across four API models.

- Task: classify each tweet as `1` ethical risk discourse or `0` not ethical discourse
- Few-shot examples: fixed 20 examples, 10 per class
- Test set: all rows from validation dataset
- Outputs: per-model prediction CSV files and one model comparison CSV under `results/`


## 1. Install Required Libraries

Run this cell if the environment does not already have the required SDKs and analysis libraries installed.


In [ ]:
# %pip install -q pandas tqdm scikit-learn openai anthropic google-genai

## 2. Imports

In [ ]:
from __future__ import annotations

import re
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Callable

import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
)

from openai import OpenAI
from anthropic import Anthropic
from google import genai
from google.genai import types

## 3. API Keys

Fill in the API keys before running the API-backed prediction cells. Do not hard-code real keys in a committed copy of this notebook.


In [ ]:
# =========================
# API KEYS - FILL BEFORE RUN
# =========================

OPENAI_API_KEY = "YOUR_OPENAI_API_KEY_HERE"
ANTHROPIC_API_KEY = "YOUR_ANTHROPIC_API_KEY_HERE"
GOOGLE_API_KEY = "YOUR_GOOGLE_API_KEY_HERE"

## 4. Configuration

In [ ]:
DATA_PATH = Path("fewshot_validation_dataset.csv") # VALIDATION DATASET
FEWSHOT_DATA_PATH = Path("fewshot_examples_20.csv") # FEWSHOT Example
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

EXPECTED_N_ROWS = 385 # Validation Set Size
EXPECTED_FEWSHOT_N = 20 # Fewshot Sample Size
EXPECTED_TEST_N = 385 # Test SAMPLE SIZE

MAX_RETRIES = 3
INITIAL_RETRY_SLEEP_SECONDS = 2.0
API_SLEEP_SECONDS = 0.5
MAX_OUTPUT_TOKENS = 8
GOOGLE_MAX_OUTPUT_TOKENS = 1024
GOOGLE_THINKING_BUDGET = 128

DATE_TAG = datetime.now().strftime("%y%m%d")

MODEL_CONFIGS = [
    {"provider": "openai", "model_name": "gpt-4.1"},
    {"provider": "openai", "model_name": "gpt-3.5-turbo"},
    {"provider": "anthropic", "model_name": "claude-sonnet-4-6"},
    {"provider": "google", "model_name": "gemini-2.5-pro"},
]

MODEL_CONFIGS

## 5. Load CSV Data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Input CSV not found: {DATA_PATH.resolve()}\n"
    )
    
if not FEWSHOT_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Input CSV not found: {FEWSHOT_DATA_PATH.resolve()}\n"
    )

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded shape: {df_raw.shape}")
df_raw.head()

## 6. Validate Required Columns

In [ ]:
REQUIRED_COLUMNS = ["content", "Gold_label"]
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df_raw[REQUIRED_COLUMNS].copy()
df["content"] = df["content"].fillna("").astype(str)
df["Gold_label"] = df["Gold_label"].astype(int)

label_values = set(df["Gold_label"].unique())
if not label_values.issubset({0, 1}):
    raise ValueError(f"Gold_label must contain only 0 and 1. Found: {sorted(label_values)}")

if len(df) != EXPECTED_N_ROWS:
    print(f"Warning: expected {EXPECTED_N_ROWS} rows, but found {len(df)} rows.")

label_counts = df["Gold_label"].value_counts().sort_index()
if (label_counts < 10).any():
    raise ValueError(f"Each class must have at least 10 rows. Label counts:\n{label_counts}")

label_counts

## 7. Load Few-shot Examples

In [ ]:
fewshot_df = pd.read_csv(FEWSHOT_DATA_PATH).reset_index(drop=False)

if len(fewshot_df) != EXPECTED_FEWSHOT_N:
    print(f"Warning: expected {EXPECTED_FEWSHOT_N} test rows, but found {len(fewshot_df)} rows.")

fewshot_df = fewshot_df[["index", "content", "Gold_label"]]

## 8. Create Test Set

In [ ]:
test_df = df[["content", "Gold_label"]].reset_index(drop=True)

if len(test_df) != EXPECTED_TEST_N:
    print(f"Warning: expected {EXPECTED_TEST_N} test rows, but found {len(test_df)} rows.")

print(f"Few-shot examples: {len(fewshot_df)}")
print(f"Test samples: {len(test_df)}")
test_df.head()

## 9. Save Few-shot Examples

In [ ]:
fewshot_save_df = fewshot_df[["content", "Gold_label"]].copy()
fewshot_save_df["split"] = "fewshot_example"

fewshot_output_path = RESULTS_DIR / f"{DATE_TAG}_fewshot_examples.csv"
fewshot_save_df.to_csv(fewshot_output_path, index=False, encoding="utf-8-sig")

print(f"Saved few-shot examples to: {fewshot_output_path}")
fewshot_save_df.head()

## 10. Build Prompt Function

In [ ]:
def build_fewshot_prompt(tweet: str, fewshot_df: pd.DataFrame) -> str:
    examples_text = []
    for i, row in fewshot_df.reset_index(drop=True).iterrows():
        examples_text.append(
            f'Example {i + 1}\n'
            f'Tweet:\n"""{row["content"]}"""\n'
            f'Label: {int(row["Gold_label"])}'
        )

    examples_block = "\n\n".join(examples_text)

    prompt = f"""You are a strict binary classifier.

    Classify the tweet as:

    1 = Ethical risk discourse
    0 = Not ethical discourse

    Ethical risk discourse refers to concerns about risks or harms related to generative AI in one or more of the following areas:

    1. Technical safety
       (e.g., hallucination causing harm, unsafe outputs, lack of explainability)

    2. Privacy or data misuse
       (e.g., personal data leakage, unauthorized data scraping, confidentiality breaches)

    3. Fairness or discrimination
       (e.g., biased outputs, discrimination, copyright or creator rights violations)

    4. Malicious misuse
       (e.g., deepfake fraud, misinformation, jailbreak exploitation, criminal misuse)

    5. Societal or democratic risks
       (e.g., job displacement framed as harm, threats to democracy, digital divide, human rights concerns)

    Tweets that are general usage, praise, humor, or neutral descriptions without risk implications should be labeled 0.

    Below are labeled examples.

    {examples_block}

    Now classify the following tweet.

    Tweet:
    \"\"\"{tweet}\"\"\"

    Return only 0 or 1."""

    return prompt

In [ ]:
sample_prompt = build_fewshot_prompt(test_df.loc[0, "content"], fewshot_df)
print(sample_prompt[:1500])
print("...\n")
print(sample_prompt[-500:])

## 11. Response Parser

In [ ]:
def parse_binary_response(raw_response: str):
    text = str(raw_response).strip()

    if text in {"0", "1"}:
        return int(text), True

    match = re.search(r"[01]", text)
    if match:
        return int(match.group(0)), True

    return "invalid", False


def convert_invalid_to_wrong_label(y_true, pred_label):
    if pred_label in [0, 1]:
        return pred_label
    return 1 - int(y_true)

## 12. Initialize API Clients

In [ ]:
def require_api_key(api_key: str, key_name: str) -> str:
    if not api_key or api_key.startswith("YOUR_"):
        raise ValueError(f"Fill {key_name} before running API prediction cells.")
    return api_key


openai_client = OpenAI(api_key=require_api_key(OPENAI_API_KEY, "OPENAI_API_KEY"))
anthropic_client = Anthropic(api_key=require_api_key(ANTHROPIC_API_KEY, "ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=require_api_key(GOOGLE_API_KEY, "GOOGLE_API_KEY"))

print("API clients initialized.")

## 13. API Call Functions

In [ ]:
def retry_api_call(callable_fn: Callable[[], Any], *, max_retries: int = MAX_RETRIES) -> Any:
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            return callable_fn()
        except Exception as exc:
            last_error = exc
            if attempt == max_retries:
                break
            sleep_seconds = INITIAL_RETRY_SLEEP_SECONDS * (2 ** (attempt - 1))
            print(
                f"API call failed on attempt {attempt}/{max_retries}: "
                f"{type(exc).__name__}: {exc}. Retrying in {sleep_seconds:.1f}s."
            )
            time.sleep(sleep_seconds)
    raise last_error


def call_openai_model(model_name: str, prompt: str) -> str:
    response = retry_api_call(
        lambda: openai_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=MAX_OUTPUT_TOKENS,
        )
    )
    return (response.choices[0].message.content or "").strip()


def call_anthropic_model(model_name: str, prompt: str) -> str:
    response = retry_api_call(
        lambda: anthropic_client.messages.create(
            model=model_name,
            max_tokens=MAX_OUTPUT_TOKENS,
            messages=[{"role": "user", "content": prompt}],
        )
    )
    text_blocks = [
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text" and getattr(block, "text", None) is not None
    ]
    return "".join(text_blocks).strip()

def extract_gemini_text(response) -> str:
    if getattr(response, "text", None):
        return response.text.strip()

    text_parts = []
    for candidate in getattr(response, candidates, []) or []:
        content = getattr(candidate, "content", None)
        for part in getattr(content, "parts", []) or []:
            text = getattr(part, "text", None)
            if text:
                text_parts.append(text)
    return "".join(text_parts).strip()

def summarize_gemini_empty_response(response) -> str:
    finish_reasons = [
        str(getattr(candidate, "finish_reason", None))
        for candidate in getattr(response, "candidates", []) or []
    ]
    usage = getattr(response, "usage_metadata", None)
    return f"Empty Gemini response. finish_reasons={finish_reasons}; usage_metadata={usage}"


def call_google_model(model_name: str, prompt: str) -> str:
    response = retry_api_call(
        lambda: gemini_client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0,
                max_output_tokens=GOOGLE_MAX_OUTPUT_TOKENS,
                thinking_config=types.ThinkingConfig(
                    thinking_budget=GOOGLE_THINKING_BUDGET,
                )
            ),
        )
    )
    text = extract_gemini_text(response)
    if not text:
        raise RuntimeError(summarize_gemini_empty_response(response))
    return text


def call_model(provider: str, model_name: str, prompt: str) -> str:
    if provider == "openai":
        return call_openai_model(model_name, prompt)
    if provider == "anthropic":
        return call_anthropic_model(model_name, prompt)
    if provider == "google":
        return call_google_model(model_name, prompt)
    raise ValueError(f"Unsupported provider: {provider}")

## 14. Model Prediction Loop

In [ ]:
def predict_for_model(model_config: dict, test_df: pd.DataFrame, fewshot_df: pd.DataFrame) -> pd.DataFrame:
    provider = model_config["provider"]
    model_name = model_config["model_name"]
    records = []

    progress = tqdm(
        test_df.iterrows(),
        total=len(test_df),
        desc=f"Predicting with {model_name}",
        unit="tweet",
    )

    for processed, (_, row) in enumerate(progress, start=1):
        prompt = build_fewshot_prompt(row["content"], fewshot_df)

        try:
            raw_response = call_model(provider, model_name, prompt)
            pred_label, is_valid_response = parse_binary_response(raw_response)
        except Exception as exc:
            raw_response = f"API_ERROR: {type(exc).__name__}: {exc}"
            pred_label = "invalid"
            is_valid_response = False

        records.append(
            {
                "content": row["content"],
                "Gold_label": int(row["Gold_label"]),
                "pred_label": pred_label,
                "raw_response": raw_response,
                "is_valid_response": bool(is_valid_response),
                "model_name": model_name,
                "split": "test",
            }
        )

        progress.set_postfix(
            processed=processed,
            remaining=len(test_df) - processed,
            valid=sum(record["is_valid_response"] for record in records),
            invalid=sum(not record["is_valid_response"] for record in records),
        )
        time.sleep(API_SLEEP_SECONDS)

    return pd.DataFrame(records)


def prediction_output_path(model_name: str) -> Path:
    return RESULTS_DIR / f"{DATE_TAG}_{model_name}_predictions.csv"


## 15. Evaluation Metrics

In [ ]:
def compute_metrics(prediction_df: pd.DataFrame) -> dict:
    y_true = prediction_df["Gold_label"].astype(int).tolist()
    y_pred_for_metric = [
        convert_invalid_to_wrong_label(y_true_value, pred_label)
        for y_true_value, pred_label in zip(y_true, prediction_df["pred_label"].tolist())
    ]

    invalid_count = int((~prediction_df["is_valid_response"].astype(bool)).sum())
    n_test = len(prediction_df)

    return {
        "model_name": prediction_df["model_name"].iloc[0],
        "accuracy": accuracy_score(y_true, y_pred_for_metric),
        "precision": precision_score(y_true, y_pred_for_metric, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred_for_metric, pos_label=1, zero_division=0),
        "f1_score": f1_score(y_true, y_pred_for_metric, pos_label=1, zero_division=0),
        "cohens_k": cohen_kappa_score(y_true, y_pred_for_metric),
        "invalid_count": invalid_count,
        "invalid_rate": invalid_count / n_test if n_test else 0,
        "n_test": n_test,
    }

## 16. Run All Four Models and Save Outputs

This cell sends API requests for every test tweet and each configured model. It may take time and incur API costs.


In [ ]:
all_prediction_dfs = []
comparison_rows = []

for model_config in MODEL_CONFIGS:
    model_name = model_config["model_name"]
    print(f"\n=== Starting predictions for {model_name} ===")

    prediction_df = predict_for_model(model_config, test_df, fewshot_df)
    output_path = prediction_output_path(model_name)
    prediction_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved predictions to: {output_path}")

    metrics = compute_metrics(prediction_df)
    comparison_rows.append(metrics)
    all_prediction_dfs.append(prediction_df)

    display(pd.DataFrame([metrics]))

comparison_df = pd.DataFrame(comparison_rows)
comparison_output_path = RESULTS_DIR / f"{DATE_TAG}_model_comparison.csv"
comparison_df.to_csv(comparison_output_path, index=False, encoding="utf-8-sig")

print(f"\nSaved model comparison to: {comparison_output_path}")
comparison_df

## 17. Display Final Comparison Table

In [ ]:
comparison_df.sort_values(by="f1_score", ascending=False).reset_index(drop=True)